# RT-DETR (L) Drone Detection Training

**Dataset**: 1,359 images, 1 class (drone)
**Training Time**: ~8-12 hours on Mac M4
**Advantage**: Higher accuracy than YOLOv8s

## 0. Setup Display

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✓ Display setup complete')

## 1. Install Ultralytics

In [ ]:
!pip install ultralytics

## 2. Load RT-DETR-L Model

In [ ]:
from ultralytics import RTDETR

# Load pretrained RT-DETR-L model
model = RTDETR('rtdetr-l.pt')

print('✓ RT-DETR-L model loaded')

## 3. Train Model

Optimized for Mac M4: batch=8 (transformers need more memory)

In [ ]:
# Training configuration  
results = model.train(
    data='/Users/hilkin/Development/Drone-Detection/drone_dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=8,          # RT-DETR needs more memory
    device='mps',     # Mac M4 GPU
    workers=8,
    cache='ram',      # Cache in RAM
    project='rtdetr_drone_results',
    name='training',
    patience=20,
    save_period=10,
    amp=True,
    plots=True,
    lr0=0.0001,       # Lower LR for transformers
    lrf=0.01
)

print('\n✓ Training complete!')

## 4. Evaluate Model

In [ ]:
# Evaluate on validation set
metrics = model.val()

print(f'\nmAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall: {metrics.box.mr:.4f}')

## 5. Benchmark Inference Speed

In [ ]:
import time
import glob

# Get test images
test_imgs = glob.glob('/Users/hilkin/Development/Drone-Detection/drone-dataset-(uav)-DatasetNinja/valid/images/*')[:100]

# Warm up
_ = model.predict(test_imgs[0], verbose=False)

# Benchmark
start = time.time()
for img in test_imgs:
    _ = model.predict(img, verbose=False)
end = time.time()

fps = len(test_imgs) / (end - start)
print(f'\nFPS: {fps:.2f}')
print(f'ms/image: {(end - start) / len(test_imgs) * 1000:.2f}ms')

## 6. Visualize Predictions (with Image Display)

In [ ]:
import cv2
import numpy as np

# Run predictions on sample images
sample_imgs = test_imgs[:6]
results = model.predict(sample_imgs, conf=0.25, save=False)

# Display with larger figure
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (result, img_path) in enumerate(zip(results, sample_imgs)):
    # Get plotted image
    img_bgr = result.plot()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Display
    axes[idx].imshow(img_rgb)
    axes[idx].axis('off')
    axes[idx].set_title(f'{len(result.boxes)} drone(s) detected', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('✓ Predictions displayed above')

## 7. Show Training Results

In [ ]:
import os
from IPython.display import Image as IPImage, display

# Display training results
results_img = 'rtdetr_drone_results/training/results.png'

if os.path.exists(results_img):
    print("Training Results:")
    display(IPImage(filename=results_img, width=1000))
else:
    print("Results image not found yet. Continue training!")

## 8. Show Confusion Matrix

In [ ]:
# Display confusion matrix
cm_img = 'rtdetr_drone_results/training/confusion_matrix.png'

if os.path.exists(cm_img):
    print("Confusion Matrix:")
    display(IPImage(filename=cm_img, width=600))
else:
    print("Confusion matrix not found yet.")

## 9. Export Model

In [ ]:
# Export to different formats
model.export(format='onnx')
model.export(format='coreml')

print('✓ Model exported!')

## 10. Test on Your Own Image (Optional)

In [ ]:
# Test on your own image
test_image_path = '/Users/hilkin/Development/Drone-Detection/pexels.jpeg'  # Change this

if os.path.exists(test_image_path):
    # Run prediction
    result = model.predict(test_image_path, conf=0.25)[0]
    
    # Display
    plt.figure(figsize=(12, 8))
    img_bgr = result.plot()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'RT-DETR: Detected {len(result.boxes)} drone(s)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f'✓ Found {len(result.boxes)} drone(s)')
else:
    print(f'Image not found: {test_image_path}')

## Summary

**Model Location**: `rtdetr_drone_results/training/weights/best.pt`

**RT-DETR Advantages**:
- Higher accuracy (88-95% mAP50 vs 85-92% for YOLOv8s)
- Better at detecting small objects
- More robust to occlusions  
- Transformer-based architecture

**Performance**: Check metrics above

**To use your model later**:
```python
from ultralytics import RTDETR
model = RTDETR('rtdetr_drone_results/training/weights/best.pt')
results = model.predict('your_image.jpg')
```